# Response caching: answer repeated questions without calling the model

**The idea.** In most LLM apps a few questions make up most of the traffic (the 80/20 rule): everybody asks about the same famous people, the same return policy, the same bestsellers. Once the model has answered a question, keep the answer. When the same question, or one that means the same thing, comes in again, reply from the cache. That takes milliseconds and costs nothing.

**The agent** writes a 2-3 sentence profile of a person. A small LangGraph graph puts two caches in front of the model:

```
query → check exact cache ──hit──→ reply
            │ miss
            ▼
        check semantic cache ──hit──→ reply
            │ miss
            ▼
        call the model API → store the answer → reply
```

- **Exact cache:** the same text, ignoring case and punctuation ("Marie Curie" = "marie curie!"). A dictionary lookup.
- **Semantic cache:** a question that means the same thing ("Tell me about Elon" ≈ "Elon Musk"). The query is turned into an embedding vector and compared with the cached ones.

**Two design questions, each tested here:**
1. **How long should an answer live in the cache?** (section 5, time to live)
2. **What counts as a "similar" question?** (sections 6 and 7: similar-looking questions about *different* people, and a guard against them)

**Before you run it:** it needs `CEREBRAS_API_KEY` (or `GROQ_API_KEY`) in a `.env` file at the repository root (see the README) and makes about 15 short model calls (roughly 6,000 tokens). Section 8's traffic simulation makes no API calls.

In [1]:
import os
import re
import textwrap
import time
import warnings
from typing import TypedDict

os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")  # keep the model-loading bars out of the output
os.environ.setdefault("TRANSFORMERS_VERBOSITY", "error")
os.environ.setdefault("HF_HUB_VERBOSITY", "error")  # hides the "unauthenticated requests" notice

warnings.filterwarnings("ignore", message="IProgress not found")  # harmless notice from tqdm

import numpy as np
import pandas as pd
from dotenv import load_dotenv
from langgraph.graph import END, START, StateGraph
from openai import OpenAI
from sentence_transformers import SentenceTransformer

load_dotenv()  # finds the .env file at the repository root (it searches parent folders)

# gpt-oss-120b on Cerebras if that key is set, otherwise on Groq (both speak the OpenAI API).
if os.getenv("CEREBRAS_API_KEY"):
    PROVIDER, MODEL = "Cerebras", "gpt-oss-120b"
    client = OpenAI(base_url="https://api.cerebras.ai/v1", api_key=os.environ["CEREBRAS_API_KEY"], max_retries=10)
elif os.getenv("GROQ_API_KEY"):
    PROVIDER, MODEL = "Groq", "openai/gpt-oss-120b"
    client = OpenAI(base_url="https://api.groq.com/openai/v1", api_key=os.environ["GROQ_API_KEY"], max_retries=10)
else:
    raise RuntimeError("Put CEREBRAS_API_KEY or GROQ_API_KEY in a .env file at the repository root.")

encoder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")  # small local embedding model
print(f"Model: {MODEL} on {PROVIDER}")

Model: gpt-oss-120b on Cerebras


## 1. The expensive step: one model call

Everything the cache saves is this call. A counter records the calls, tokens and time, so we can see what the cache saved.

In [2]:
USAGE = {"calls": 0, "tokens": 0, "seconds": 0.0}

SYSTEM = ("You write short, factual profiles of well-known people. Reply in 2-3 plain sentences about the "
          "person the user asks about. If you are not sure who they mean, say so in one sentence.")


def ask_model(query: str) -> str:
    start = time.perf_counter()
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "system", "content": SYSTEM}, {"role": "user", "content": query}],
        temperature=0,
        max_tokens=1024,          # gpt-oss thinks before answering, and the thinking uses this budget too
        reasoning_effort="low",
    )
    USAGE["calls"] += 1
    USAGE["tokens"] += response.usage.total_tokens
    USAGE["seconds"] += time.perf_counter() - start
    answer = (response.choices[0].message.content or "").strip()
    if not answer:
        raise RuntimeError(f"Empty reply for {query!r} (finish_reason={response.choices[0].finish_reason})")
    return answer

## 2. The cache

- `normalize()` builds the exact-cache key: lower case, no punctuation, no "'s".
- `core()` drops filler words such as "tell me about" or "who is" before embedding, so that "Tell me about Elon" is compared as "elon". Without this, the shared filler makes unrelated questions look alike.
- Each entry keeps the query, its embedding, the answer and the time it was stored.
- `ttl_seconds` (**time to live**) is how long an answer stays valid. Expired entries are dropped on the next lookup.
- `threshold` is the lowest similarity (cosine, from 0 to 1) that counts as "the same question". 0.60 was picked by looking at scores like the ones in section 6.
- `guard` is an extra check that section 7 switches on.

In [3]:
FILLER = {"tell", "me", "about", "info", "information", "on", "who", "is", "was", "what", "do", "you", "know",
          "the", "a", "an", "of", "please", "give", "some", "biography", "bio", "profile", "can"}


def normalize(text: str) -> str:
    """'Marie Curie!' and 'marie curie' get the same key."""
    text = text.lower().replace("’", "'")
    text = re.sub(r"'s\b", "", text)
    text = re.sub(r"[^a-z0-9 ]+", " ", text)
    return " ".join(text.split())


def core(text: str) -> str:
    """The words that carry the meaning: 'Tell me about Elon' -> 'elon'."""
    words = [w for w in normalize(text).split() if w not in FILLER]
    return " ".join(words) or normalize(text)


class ResponseCache:
    def __init__(self, encoder, threshold=0.60, ttl_seconds=24 * 3600, guard=False):
        self.encoder, self.threshold, self.ttl, self.guard = encoder, threshold, ttl_seconds, guard
        self.entries = {}  # normalized query -> {"query", "core", "vector", "answer", "created"}

    def _drop_expired(self):
        now = time.time()
        self.entries = {key: e for key, e in self.entries.items() if now - e["created"] < self.ttl}

    def lookup_exact(self, query):
        self._drop_expired()
        return self.entries.get(normalize(query))

    def lookup_semantic(self, query):
        """The most similar cached query, if it is similar enough. Returns (entry or None, note)."""
        self._drop_expired()
        if not self.entries:
            return None, "the cache is empty"
        q_core = core(query)
        vector = self.encoder.encode(q_core, normalize_embeddings=True)
        scores = {key: float(vector @ e["vector"]) for key, e in self.entries.items()}
        best_key = max(scores, key=scores.get)
        best, score = self.entries[best_key], scores[best_key]
        if score < self.threshold:
            return None, f"closest: {best['query']!r} at {score:.2f}, below the {self.threshold} threshold"
        if self.guard:
            extra = [w for w in q_core.split() if w not in best["core"].split()]
            if extra:
                return None, f"{best['query']!r} scores {score:.2f}, but the guard rejects it: {' '.join(extra)!r} is not in it"
        return best, f"similar to {best['query']!r} ({score:.2f})"

    def add(self, query, answer):
        q_core = core(query)
        self.entries[normalize(query)] = {
            "query": query, "core": q_core, "answer": answer, "created": time.time(),
            "vector": self.encoder.encode(q_core, normalize_embeddings=True),
        }

## 3. The agent graph

Four nodes: check the exact cache, check the semantic cache, call the model, store the answer. After each cache check, a hit goes straight to the end. `answer_fn` is the model call; section 8 swaps in a fake one to simulate traffic for free.

In [4]:
class CacheState(TypedDict, total=False):
    query: str
    answer: str
    source: str   # "exact cache", "semantic cache" or "model API"
    note: str     # why the semantic cache did or did not match


def build_agent(cache: ResponseCache, answer_fn=ask_model):
    def check_exact_cache(state):
        entry = cache.lookup_exact(state["query"])
        if entry:
            return {"answer": entry["answer"], "source": "exact cache", "note": f"same text as {entry['query']!r}"}
        return {}

    def check_semantic_cache(state):
        entry, note = cache.lookup_semantic(state["query"])
        if entry:
            return {"answer": entry["answer"], "source": "semantic cache", "note": note}
        return {"note": note}

    def call_api(state):
        return {"answer": answer_fn(state["query"]), "source": "model API"}

    def store_in_cache(state):
        cache.add(state["query"], state["answer"])
        return {}

    def hit_or_miss(state):
        return "hit" if state.get("source") else "miss"

    graph = StateGraph(CacheState)
    for node in (check_exact_cache, check_semantic_cache, call_api, store_in_cache):
        graph.add_node(node.__name__, node)
    graph.add_edge(START, "check_exact_cache")
    graph.add_conditional_edges("check_exact_cache", hit_or_miss, {"hit": END, "miss": "check_semantic_cache"})
    graph.add_conditional_edges("check_semantic_cache", hit_or_miss, {"hit": END, "miss": "call_api"})
    graph.add_edge("call_api", "store_in_cache")
    graph.add_edge("store_in_cache", END)
    return graph.compile()


def run_queries(agent, queries, show_answers=True):
    rows = []
    for q in queries:
        start = time.perf_counter()
        out = agent.invoke({"query": q})
        ms = (time.perf_counter() - start) * 1000
        print(f"[{out['source']:>14}] {ms:7.1f} ms   {q}")
        if out.get("note"):
            print(f"{'':30}({out['note']})")
        if show_answers:
            print(textwrap.fill(out["answer"], 110, initial_indent=" " * 30, subsequent_indent=" " * 30))
        rows.append({"query": q, "served from": out["source"], "ms": round(ms, 1)})
    return pd.DataFrame(rows)


cache = ResponseCache(encoder)
agent = build_agent(cache)
print(agent.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	check_exact_cache(check_exact_cache)
	check_semantic_cache(check_semantic_cache)
	call_api(call_api)
	store_in_cache(store_in_cache)
	__end__([<p>__end__</p>]):::last
	__start__ --> check_exact_cache;
	call_api --> store_in_cache;
	check_exact_cache -. &nbsp;hit&nbsp; .-> __end__;
	check_exact_cache -. &nbsp;miss&nbsp; .-> check_semantic_cache;
	check_semantic_cache -. &nbsp;hit&nbsp; .-> __end__;
	check_semantic_cache -. &nbsp;miss&nbsp; .-> call_api;
	store_in_cache --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



## 4. A first run

Seven questions, including a repeat, a lowercase repeat and two rephrasings.

In [5]:
QUERIES = ["Elon Musk", "Elon Musk", "Tell me about Elon", "Marie Curie", "marie curie", "Info on Curie", "Jensen Huang"]

USAGE.update(calls=0, tokens=0, seconds=0.0)
start = time.perf_counter()
results = run_queries(agent, QUERIES)
total = time.perf_counter() - start

per_call = USAGE["seconds"] / USAGE["calls"]
print(f"\nModel calls: {USAGE['calls']} for {len(QUERIES)} questions "
      f"({len(QUERIES) - USAGE['calls']} answered from the cache)")
print(f"Total time: {total:.1f} s. Without a cache, about {len(QUERIES) * per_call:.1f} s "
      f"({len(QUERIES)} calls x {per_call:.2f} s average).")
print(f"Tokens: {USAGE['tokens']:,}. Without a cache, about {len(QUERIES) * USAGE['tokens'] / USAGE['calls']:,.0f}.")
results.groupby("served from")["ms"].agg(["count", "median"]).rename(columns={"median": "median ms"})

[     model API]   628.1 ms   Elon Musk
                              (the cache is empty)
                              Elon Musk is a South African‑born entrepreneur and engineer who co‑founded
                              PayPal, then went on to launch SpaceX in 2002 and Tesla, Inc. in 2004, becoming
                              its CEO and product architect. He is also known for ventures such as Neuralink,
                              The Boring Company, and for promoting the concept of a high‑speed transportation
                              system called the Hyperloop.
[   exact cache]     1.6 ms   Elon Musk
                              (same text as 'Elon Musk')
                              Elon Musk is a South African‑born entrepreneur and engineer who co‑founded
                              PayPal, then went on to launch SpaceX in 2002 and Tesla, Inc. in 2004, becoming
                              its CEO and product architect. He is also known for ventures such as Neu

,count,median ms
served from,,
exact cache,2,1.5
model API,3,628.1
semantic cache,2,13.7


**What this shows:** 3 model calls for 7 questions.
- **Speed:** an exact hit takes about 1.5 ms, and a semantic hit about 11-15 ms (mostly embedding the query). A model call takes about 500-700 ms.
- **Total:** 1.9 s instead of about 4.1 s, and 694 tokens instead of about 1,600.

## 5. Design question 1: how long should an answer live?

A cached answer goes stale when the facts behind it change. The **time to live (TTL)** sets when an entry is thrown away:
- profiles of famous people change slowly: a day or a week is fine;
- prices, stock levels or news: minutes, or clear the entry whenever the data changes;
- anything personal ("my order status"): don't share it across users at all.

Here the TTL is set to 5 seconds to show an entry expiring.

In [6]:
short_cache = ResponseCache(encoder, ttl_seconds=5)
short_agent = build_agent(short_cache)
run_queries(short_agent, ["Sachin Tendulkar", "Sachin Tendulkar"], show_answers=False)
print("... waiting 6 seconds ...")
time.sleep(6)
run_queries(short_agent, ["Sachin Tendulkar"], show_answers=False);

[     model API]   564.2 ms   Sachin Tendulkar
                              (the cache is empty)
[   exact cache]     1.2 ms   Sachin Tendulkar
                              (same text as 'Sachin Tendulkar')
... waiting 6 seconds ...
[     model API]   651.5 ms   Sachin Tendulkar
                              (the cache is empty)


**What this shows:** the repeat inside 5 seconds came from the cache. After the wait, the entry had expired, so the model was called again.

## 6. Design question 2: what counts as a "similar" question?

The semantic cache in section 4 now holds Elon Musk, Marie Curie and Jensen Huang. Here are questions that **look** similar to those but are about **someone else**, plus one that is about Elon Musk but shares no words with "Elon Musk". They are sent through the same agent as before.

In [7]:
TRICKY = ["Pierre Curie", "Marie Curie's daughter", "Elon Musk's mother", "Jensen Ackles", "Tesla founder"]
calls_before = USAGE["calls"]
tricky_results = run_queries(agent, TRICKY)
print(f"\nModel calls: {USAGE['calls'] - calls_before}. The other {len(TRICKY) - (USAGE['calls'] - calls_before)} "
      f"were answered from the cache. Check above whose profile each one got.")

[semantic cache]    13.8 ms   Pierre Curie
                              (similar to 'Marie Curie' (0.66))
                              Marie Curie (1867–1934) was a Polish‑born physicist and chemist who pioneered
                              research on radioactivity, a term she coined. She was the first woman to win a
                              Nobel Prize, the first person to win Nobel Prizes in two different sciences
                              (Physics in 1903 and Chemistry in 1911), and she discovered the elements
                              polonium and radium. Her work laid the foundation for modern nuclear science and
                              medical radiology.
[semantic cache]    13.3 ms   Marie Curie's daughter
                              (similar to 'Marie Curie' (0.86))
                              Marie Curie (1867–1934) was a Polish‑born physicist and chemist who pioneered
                              research on radioactivity, a term she coined. She wa

**What this shows:** three of the five questions got **someone else's profile**.
- **Wrong person:** "Pierre Curie" and "Marie Curie's daughter" got Marie Curie's profile, and "Elon Musk's mother" got Elon Musk's. That is worse than no cache: the user gets a confident wrong answer, fast.
- **Near miss:** "Jensen Ackles" scored 0.56 and only just missed being served Jensen Huang's profile.
- **The opposite mistake:** "Tesla founder" is about Elon Musk but scored only 0.45, so it cost a model call. That one is harmless: the answer is right, just not free.

A higher threshold doesn't fix this, because some wrong matches score **higher** than some right ones. The table scores labelled pairs (no API calls): for each threshold, how many same-person pairs would be served from the cache (good) and how many different-person pairs would get the wrong answer (bad).

In [8]:
SAME = [("Tell me about Elon", "Elon Musk"), ("Info on Curie", "Marie Curie"), ("Who is Elon Musk?", "Elon Musk"),
        ("Musk", "Elon Musk"), ("Tell me about Jensen", "Jensen Huang"), ("info on huang", "Jensen Huang"),
        ("Tesla founder", "Elon Musk"), ("Nvidia CEO", "Jensen Huang")]
DIFFERENT = [("Pierre Curie", "Marie Curie"), ("Marie Curie's daughter", "Marie Curie"),
             ("Elon Musk's mother", "Elon Musk"), ("Jensen Ackles", "Jensen Huang"),
             ("Virat Kohli", "Sachin Tendulkar"), ("Venus Williams", "Serena Williams"),
             ("Michelle Obama", "Barack Obama"), ("Steve Wozniak", "Steve Jobs")]


def pair_scores(pairs):
    a = encoder.encode([core(x) for x, _ in pairs], normalize_embeddings=True)
    b = encoder.encode([core(y) for _, y in pairs], normalize_embeddings=True)
    return (a * b).sum(axis=1)


def guard_ok(x, y):
    return all(w in core(y).split() for w in core(x).split())


same_s, diff_s = pair_scores(SAME), pair_scores(DIFFERENT)
print("Scores, same person:     ", ", ".join(f"{x!r} {s:.2f}" for (x, _), s in zip(SAME, same_s)))
print("Scores, different person:", ", ".join(f"{x!r} {s:.2f}" for (x, _), s in zip(DIFFERENT, diff_s)))
rows = []
for t in [0.5, 0.6, 0.7, 0.8, 0.9]:
    rows.append({
        "threshold": t,
        f"same person served from cache (of {len(SAME)})": int((same_s >= t).sum()),
        f"WRONG person served (of {len(DIFFERENT)})": int((diff_s >= t).sum()),
        "same person, with guard": sum(s >= t and guard_ok(x, y) for (x, y), s in zip(SAME, same_s)),
        "WRONG person, with guard": sum(s >= t and guard_ok(x, y) for (x, y), s in zip(DIFFERENT, diff_s)),
    })
pd.DataFrame(rows)

Scores, same person:      'Tell me about Elon' 0.62, 'Info on Curie' 0.73, 'Who is Elon Musk?' 1.00, 'Musk' 0.84, 'Tell me about Jensen' 0.70, 'info on huang' 0.83, 'Tesla founder' 0.45, 'Nvidia CEO' 0.25
Scores, different person: 'Pierre Curie' 0.66, "Marie Curie's daughter" 0.86, "Elon Musk's mother" 0.80, 'Jensen Ackles' 0.56, 'Virat Kohli' 0.65, 'Venus Williams' 0.80, 'Michelle Obama' 0.74, 'Steve Wozniak' 0.58


,threshold,same person served from cache (of 8),WRONG person served (of 8),"same person, with guard","WRONG person, with guard"
0,0.5,6,8,6,0
1,0.6,6,6,6,0
2,0.7,5,4,5,0
3,0.8,3,1,3,0
4,0.9,1,0,1,0


**What this shows:** no threshold separates right matches from wrong ones.
- **At 0.6:** 6 same-person pairs hit, but so do 6 different-person pairs.
- **At 0.8:** only 1 wrong match is left, but half the right matches are lost too (3 of 6). "Tell me about Elon" scores 0.62, while "Venus Williams" against "Serena Williams" scores 0.80.
- **With the guard:** no threshold serves a wrong answer, and 0.5-0.6 still keeps the 6 right matches.
- **Descriptions never match:** "Nvidia CEO" scores 0.25. Catching those would need another step, for example asking a small model to pull out the person's name first.

## 7. A guard against wrong matches

For this agent, a question is about the same person only if it adds nothing new. The **guard** rejects a semantic match when the new question has a meaningful word that the cached question doesn't: "Pierre", "daughter", "mother", "Ackles". "Tell me about Elon" passes, because "elon" is in "Elon Musk".

This rule is specific to this app. A shop's FAQ bot would need a different one. The general lesson: decide what "the same question" means for your app and test it on labelled pairs like the ones above, rather than trusting one similarity number.

Below, a fresh cache with the guard runs section 4's questions and then the tricky ones.

In [9]:
guarded_cache = ResponseCache(encoder, guard=True)
guarded_agent = build_agent(guarded_cache)
calls_before = USAGE["calls"]
guarded = run_queries(guarded_agent, QUERIES + TRICKY, show_answers=False)
print(f"\nModel calls: {USAGE['calls'] - calls_before} for {len(guarded)} questions")

[     model API]   460.5 ms   Elon Musk
                              (the cache is empty)
[   exact cache]     1.5 ms   Elon Musk
                              (same text as 'Elon Musk')
[semantic cache]    14.9 ms   Tell me about Elon
                              (similar to 'Elon Musk' (0.62))
[     model API]   488.0 ms   Marie Curie
                              (closest: 'Elon Musk' at 0.16, below the 0.6 threshold)
[   exact cache]     1.2 ms   marie curie
                              (same text as 'Marie Curie')
[semantic cache]    10.9 ms   Info on Curie
                              (similar to 'Marie Curie' (0.73))
[     model API]   582.8 ms   Jensen Huang
                              (closest: 'Elon Musk' at 0.32, below the 0.6 threshold)
[     model API]   430.4 ms   Pierre Curie
                              ('Marie Curie' scores 0.66, but the guard rejects it: 'pierre' is not in it)
[     model API]   523.2 ms   Marie Curie's daughter
                              ('

**What this shows:** section 4's questions take the same paths as before. All five tricky questions now go to the model (8 calls for 12 questions), and each note says why the guard turned the match down.

## 8. Why it pays: simulated traffic

A quick simulation with **no API calls**: 300 requests about 20 people, where a few people are far more popular than the rest, as in real traffic. Each request uses one of several phrasings ("Elon Musk", "who is Elon Musk?", "info on Musk", "tell me about Elon"…). A fake model stands in for the real one; it "answers" with the right person's name, so we can count wrong answers served from the cache.

In [10]:
PEOPLE = ["Elon Musk", "Shah Rukh Khan", "Taylor Swift", "Sachin Tendulkar", "Marie Curie", "Lionel Messi",
          "Albert Einstein", "Virat Kohli", "Barack Obama", "Jensen Huang", "Serena Williams", "Satya Nadella",
          "Sundar Pichai", "Steve Jobs", "Ada Lovelace", "Lisa Su", "Priyanka Chopra", "Nelson Mandela",
          "Frida Kahlo", "Usain Bolt"]
PHRASINGS = ["{full}", "{lower}", "Who is {full}?", "Tell me about {full}", "{full} biography",
             "Info on {last}", "Tell me about {first}"]

rng = np.random.default_rng(0)
popularity = 1 / np.arange(1, len(PEOPLE) + 1) ** 1.5   # person 1 is asked about most, person 20 least
popularity /= popularity.sum()
requests = []
for i in rng.choice(len(PEOPLE), size=300, p=popularity):
    name = PEOPLE[i]
    phrasing = PHRASINGS[rng.integers(len(PHRASINGS))]
    requests.append((name, phrasing.format(full=name, lower=name.lower(), first=name.split()[0], last=name.split()[-1])))
top4 = sum(name in PEOPLE[:4] for name, _ in requests) / len(requests)
print(f"{len(requests)} requests; the 4 most popular people (20% of them) get {top4:.0%} of the requests")

asked_about = {q: name for name, q in requests}
rows = []
for guard in (False, True):
    sim_cache = ResponseCache(encoder, guard=guard)
    fake_calls = []

    def fake_model(query):
        fake_calls.append(query)
        return asked_about[query]  # a perfect "model": answers with the right person

    sim_agent = build_agent(sim_cache, answer_fn=fake_model)
    sources, wrong = [], []
    for name, q in requests:
        out = sim_agent.invoke({"query": q})
        sources.append(out["source"])
        if out["answer"] != name:
            wrong.append(f"{q!r} got {out['answer']}")
    rows.append({"guard": "on" if guard else "off", "model calls": len(fake_calls),
                 "exact hits": sources.count("exact cache"), "semantic hits": sources.count("semantic cache"),
                 "hit rate": f"{1 - len(fake_calls) / len(requests):.0%}", "wrong answers served": len(wrong)})
    if wrong:
        print(f"Guard {'on' if guard else 'off'}, wrong answers: " + "; ".join(sorted(set(wrong))))

summary = pd.DataFrame(rows)
saved = len(requests) - summary.loc[1, "model calls"]
print(f"\nWith the guard on, the cache saves {saved} of {len(requests)} model calls: at this notebook's measured "
      f"averages, about {saved * per_call:.0f} s of waiting and {saved * USAGE['tokens'] / USAGE['calls']:,.0f} tokens.")
summary

300 requests; the 4 most popular people (20% of them) get 70% of the requests
Guard off, wrong answers: 'Info on Kohli' got Sachin Tendulkar; 'Tell me about Virat Kohli' got Sachin Tendulkar; 'Virat Kohli biography' got Sachin Tendulkar; 'Virat Kohli' got Sachin Tendulkar; 'Who is Virat Kohli?' got Sachin Tendulkar

With the guard on, the cache saves 273 of 300 model calls: at this notebook's measured averages, about 158 s of waiting and 65,283 tokens.


,guard,model calls,exact hits,semantic hits,hit rate,wrong answers served
0,off,22,78,200,93%,5
1,on,27,77,196,91%,0


**What this shows**
- **High hit rate:** the 4 most popular people got 70% of the requests, and the cache answered 91% of all requests (27 model calls for 300, guard on).
- **Most hits are semantic (196), not exact (77).** People phrase the same question in many ways, so an exact-only cache would miss most of the savings.
- **Without the guard, every request about Virat Kohli (5) got Sachin Tendulkar's profile.**
  - The two cricketers' names score 0.65.
  - A cache hit is never stored, so Virat Kohli never got an entry of his own.
- **The guard's cost and benefit:** it costs 5 extra model calls and removes every wrong answer.

## Summary

- **Check an exact cache first (about 1-2 ms), then a semantic cache (about 10-20 ms).** Both are far cheaper than a model call: about half a second here, and often several seconds with bigger models and longer answers.
- **Set the time to live by how fast the facts change,** and never share personal answers (orders, accounts) across users.
- **Deciding what counts as "similar" is the hard part.** A similarity threshold alone served wrong-person answers at every setting that still caught rephrasings. Test on labelled pairs, and add a rule that fits your app (here, the guard).
- **In production,** the cache usually lives in Redis or a vector database rather than a Python dictionary. The cache key should include the model and prompt version, so that a prompt change doesn't serve old answers.